## Parallel Workflow with Microsoft Agent Framework and Microsoft Foundry

**Fundamental logic:** A parallel workflow runs independent tasks concurrently and then combines their results at a synchronization point.


![parallel-workflow](./Assets/parallel_workflow.png)

**Fundamental logic:** The diagram illustrates fan-out from location selection and fan-in before itinerary planning.


In [1]:
# Fundamental logic: Pinned packages provide the workflow, Azure client, and environment-loading APIs used throughout
# the notebook.

%pip install agent-framework==1.0.0b251209 python-dotenv azure-ai-projects==2.0.0b2

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Setting Up the Environment

**Fundamental logic:** This setup cell imports workflow primitives and loads the Foundry project and model configuration.


In [2]:
# Fundamental logic: WorkflowBuilder defines graph edges, Executor defines work nodes, and WorkflowContext carries
# messages between those nodes.

import os
from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from agent_framework import Executor, WorkflowBuilder, WorkflowContext, WorkflowOutputEvent, handler
from agent_framework import WorkflowBuilder, WorkflowViz

load_dotenv()
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model = os.getenv("AI_FOUNDRY_DEPLOYMENT_NAME")

print("Project Endpoint: ", project_endpoint)
print("Model: ", model)

Project Endpoint:  https://ajay-agent-project111-resource.services.ai.azure.com/api/projects/ajay-agent-project111
Model:  ajay-gpt-4o


### Defining a Function to Create a Chat Agent

**Fundamental logic:** The helper creates reusable specialist agents while keeping authentication and conversation setup in one place.


In [3]:
# Fundamental logic: Each agent gets a separate conversation and instruction prompt, allowing specialists to work
# independently in parallel.

from agent_framework import ChatAgent
from agent_framework.azure import AzureAIClient
from azure.ai.projects.aio import AIProjectClient
from azure.identity.aio import AzureCliCredential

# Cell 2: Define async workflow
async def create_agent(agent_name: str,
                       agent_instructions: str) -> ChatAgent:
    
    # Create async Azure credential
    credential = AzureCliCredential()

    # creating the Foundry Project Client
    project_client = AIProjectClient(
        endpoint=project_endpoint,
        credential=credential
    )

    # creating a conversation using the OpenAI Client
    openai_client = project_client.get_openai_client()
    conversation = await openai_client.conversations.create()
    conversation_id = conversation.id
    print("Conversation ID: ", conversation_id)
    
    # Initialize the Azure AI Agent Client
    chat_client = AzureAIClient(project_client=project_client,
                                conversation_id=conversation_id,
                                model_deployment_name=model)

    try:
        agent = chat_client.create_agent(
            name=agent_name,
            instructions=agent_instructions,
        )

        print("{} Agent created successfully!".format(agent_name))
        return agent

    finally:
        # Clean up async clients
        await chat_client.close()
        await credential.close()

### Creating the Location Picker Agent

**Fundamental logic:** The first specialist converts the original vacation preferences into a selected location.


In [4]:
# Fundamental logic: This agent is the workflow entry specialist and produces the shared input for all parallel
# branches.

location_picker_agent = await create_agent(
    agent_name = "Location-Picker-Agent",
    agent_instructions = """You are a helpful assistant that helps users pick a location for their vacation."""
)

Conversation ID:  conv_af90112eb4d3eb9c00OdTMvbxVsfXSQc4aWxewi6hzqMGRgk7U
Location-Picker-Agent Agent created successfully!


### Creating the Destination Recommender Agent

**Fundamental logic:** The destination specialist recommends attractions and places based on the selected location.


In [5]:
# Fundamental logic: This separate role keeps destination recommendations focused and independently executable.

destination_recommender_agent = await create_agent(
    agent_name = "Destination-Recommender-Agent",
    agent_instructions = """You are a travel expert that provides personalized vacation recommendations based on user preferences and locations. """
)

Conversation ID:  conv_5ec7cc2ae9dc3a5300X6RnO2K2BQA5mZKDpeVbKjcbXJadjQbW
Destination-Recommender-Agent Agent created successfully!


### Creating the Weather Agent

**Fundamental logic:** The weather specialist contributes climate and conditions relevant to planning.


In [6]:
# Fundamental logic: This agent can run concurrently because its work does not depend on destination or cuisine
# results.

weather_agent = await create_agent(
    agent_name = "Weather-Agent",
    agent_instructions = """You are a weather expert that provides accurate and up-to-date weather information for various locations selected"""
)

Conversation ID:  conv_a9b38409a81730b600qWhkrM3eZ8eiPm9MrKIyVAiqQvGmUkhv
Weather-Agent Agent created successfully!


### Creating the Cuisine Suggestion Agent

**Fundamental logic:** The cuisine specialist contributes local food and dining suggestions.


In [7]:
# Fundamental logic: This agent forms the third independent branch that receives the selected location.

cuisine_suggestion_agent = await create_agent(
    agent_name = "Cuisine-Suggestion-Agent",
    agent_instructions = """You are a culinary expert that suggests popular local cuisines and dining options based on the selected vacation destinations."""
)

Conversation ID:  conv_0b7589db451825d000paZTSjsWIwcWQ8rHj2ToJbQxmgo8Vuzj
Cuisine-Suggestion-Agent Agent created successfully!


### Creating the Itinerary Planner Agent

**Fundamental logic:** The itinerary specialist combines all parallel outputs into one coherent final plan.


In [8]:
# Fundamental logic: This downstream agent waits for destination, weather, and cuisine results before producing the
# final answer.

itinerary_planner_agent = await create_agent(
    agent_name = "Itinerary-Planner-Agent",
    agent_instructions = """You are an itinerary planning expert that creates detailed travel itineraries based on user preferences, selected destinations, weather conditions, and local cuisine options."""
)

Conversation ID:  conv_ec2a6131c1ceb86900QG41UuHAB3feij2eOfGdYECGfX3efPrt
Itinerary-Planner-Agent Agent created successfully!


### Creating the Location Selector Agent Executor

**Fundamental logic:** An executor adapts an agent into a workflow node and controls how its output is routed.


In [9]:
# Fundamental logic: The start executor sends its result as an intermediate message, triggering every connected fan-
# out branch.

class LocationSelectorExecutor(Executor):

    @handler
    async def handle(self, user_query: str, ctx: WorkflowContext[str]) -> None:
        response = await location_picker_agent.run(user_query)

        await ctx.send_message(str(response))

### Creating the Destination Recommender Agent Executor

**Fundamental logic:** This executor represents one independent destination-recommendation branch.


In [10]:
# Fundamental logic: Its handler receives the location, invokes the specialist, and sends the result toward fan-in.

class DestinationRecommenderExecutor(Executor):

    @handler
    async def handle(self, location: str, ctx: WorkflowContext[str]) -> None:
        response = await destination_recommender_agent.run(location)

        await ctx.send_message(str(response))

### Creating the Weather Agent Executor

**Fundamental logic:** This executor represents the independent weather branch.


In [11]:
# Fundamental logic: The weather output is emitted as an intermediate message for later aggregation.

class WeatherExecutor(Executor):

    @handler
    async def handle(self, location: str, ctx: WorkflowContext[str]) -> None:
        response = await weather_agent.run(location)

        await ctx.send_message(str(response))

### Creating the Cuisine Suggestion Agent Executor

**Fundamental logic:** This executor represents the independent cuisine branch.


In [12]:
# Fundamental logic: The cuisine output is emitted independently, allowing it to run concurrently with the other
# branches.

class CuisineSuggestionExecutor(Executor):

    @handler
    async def handle(self, location: str, ctx: WorkflowContext[str]) -> None:
        response = await cuisine_suggestion_agent.run(location)

        await ctx.send_message(str(response))

### Creating the Itinerary Planner Agent Executor

**Fundamental logic:** The final executor consumes the aggregated list produced after all parallel branches complete.


In [13]:
# Fundamental logic: Unlike intermediate nodes, the final node uses yield_output to publish the workflow result.

class ItineraryPlannerExecutor(Executor):

    @handler
    async def handle(self, results: list[str], ctx: WorkflowContext[str]) -> None:
        response = await itinerary_planner_agent.run(results)

        await ctx.yield_output(str(response))

### Building the Parallel Workflow

**Fundamental logic:** The graph uses fan-out for concurrency and fan-in as a barrier that waits for every required branch.


In [14]:
# Fundamental logic: Executor instances become graph nodes; edges define message routing and execution dependencies.

# Create the executor instances / objects
location_selector_executor = LocationSelectorExecutor(id="LocationSelector")
destination_recommender_executor = DestinationRecommenderExecutor(id="DestinationRecommender")
weather_executor = WeatherExecutor(id="Weather")
cuisine_suggestion_executor = CuisineSuggestionExecutor(id="CuisineSuggestion")
itinerary_planner_executor = ItineraryPlannerExecutor(id="ItineraryPlanner")

# Build the workflow
workflow = (
    WorkflowBuilder()
    .set_start_executor(location_selector_executor)
    .add_fan_out_edges(location_selector_executor, [destination_recommender_executor, weather_executor, cuisine_suggestion_executor])
    .add_fan_in_edges([destination_recommender_executor, weather_executor, cuisine_suggestion_executor], itinerary_planner_executor)
    .build()
)

viz = WorkflowViz(workflow)

Adding fan-out edges with Executor or AgentProtocol instances directly is not recommended, because workflow instances created from the builder will share the same executor/agent instances. Consider using registered names for lazy initialization instead.
Adding fan-in edges with Executor or AgentProtocol instances directly is not recommended, because workflow instances created from the builder will share the same executor/agent instances. Consider using registered names for lazy initialization instead.


### Generating Mermaid Diagram for Visualization

**Fundamental logic:** Mermaid visualization turns the workflow graph into a human-readable diagram.


In [15]:
# Fundamental logic: The graph representation is rendered inline so the fan-out and fan-in topology can be verified
# before execution.

mermaid_content = viz.to_mermaid()

# printing mermaid content as markdown
from IPython.display import Markdown, display
display(Markdown(f"```mermaid\n{mermaid_content}\n```"))

```mermaid
flowchart TD
  LocationSelector["LocationSelector (Start)"];
  DestinationRecommender["DestinationRecommender"];
  Weather["Weather"];
  CuisineSuggestion["CuisineSuggestion"];
  ItineraryPlanner["ItineraryPlanner"];
  fan_in__ItineraryPlanner__ca5ba06d((fan-in))
  CuisineSuggestion --> fan_in__ItineraryPlanner__ca5ba06d;
  DestinationRecommender --> fan_in__ItineraryPlanner__ca5ba06d;
  Weather --> fan_in__ItineraryPlanner__ca5ba06d;
  fan_in__ItineraryPlanner__ca5ba06d --> ItineraryPlanner;
  internal_LocationSelector --> LocationSelector;
  internal_DestinationRecommender --> DestinationRecommender;
  internal_Weather --> Weather;
  internal_CuisineSuggestion --> CuisineSuggestion;
  LocationSelector --> DestinationRecommender;
  LocationSelector --> Weather;
  LocationSelector --> CuisineSuggestion;
  internal_ItineraryPlanner --> ItineraryPlanner;
```

### Running the Workflow and Streaming Events

**Fundamental logic:** Streaming exposes intermediate workflow events as they occur and identifies the final output event.


In [16]:
# Fundamental logic: run_stream executes the graph asynchronously; WorkflowOutputEvent indicates that the final
# executor has yielded its result.

# Run the workflow and stream events in notebook
async def main():
    async for event in workflow.run_stream("help me plan a vacation to India with the following details: I love historical sites, prefer warm weather, and enjoy trying local foods."):
        print(f"Event: {event}")
        if isinstance(event, WorkflowOutputEvent):
            print(f"Workflow completed with result: {event.data}")

await main()

Event: WorkflowStartedEvent(origin=WorkflowEventSource.FRAMEWORK, data=None)
Event: WorkflowStatusEvent(state=WorkflowRunState.IN_PROGRESS, data=None, origin=WorkflowEventSource.FRAMEWORK)
Event: ExecutorInvokedEvent(executor_id=LocationSelector, data=help me plan a vacation to India with the following details: I love historical sites, prefer warm weather, and enjoy trying local foods.)
Event: ExecutorCompletedEvent(executor_id=LocationSelector, data=['India is an incredible destination known for its rich history, warm weather, and diverse cuisine. Here\'s a tailored vacation plan based on your preferences:\n\n---\n\n### **Duration**: 10 Days\n\n### **Season**:\n- **Best Time to Visit**: October to March (Warm but pleasant weather in most areas).\n\n---\n\n### **Suggested Itinerary**:\n\n#### **Day 1-3: Delhi**\nStart in the bustling capital city of India, which is rich in historical landmarks and has incredible street food.\n- **Historical Sites to Visit**:\n  - Red Fort: Mughal-era f